<a href="https://colab.research.google.com/github/dimalgeorge3/BERT-Transformer-based-NLP/blob/main/BERT_Transformer_based_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# =====================
# 1. Load Amazon Dataset
# =====================

raw = load_dataset("amazon_polarity")

# FIX: shuffle the raw split BEFORE selecting — dataset["train"] returns a Dataset, not DatasetDict
train_split = raw["train"].shuffle(seed=42)

train_dataset = train_split.select(range(30000))
val_dataset   = train_split.select(range(30000, 35000))
test_dataset  = train_split.select(range(35000, 38000))

# =====================
# 2. Tokenizer
# =====================

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    # FIX: merge title + content for richer signal (same as best practice for this dataset)
    texts = [t + " " + c for t, c in zip(example["title"], example["content"])]
    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset   = val_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function, batched=True)

# FIX: remove both title and content after merging
for ds in [train_dataset, val_dataset, test_dataset]:
    pass  # handled individually below (HF datasets are immutable)

train_dataset = train_dataset.remove_columns(["title", "content"])
val_dataset   = val_dataset.remove_columns(["title", "content"])
test_dataset  = test_dataset.remove_columns(["title", "content"])

train_dataset = train_dataset.rename_column("label", "labels")
val_dataset   = val_dataset.rename_column("label", "labels")
test_dataset  = test_dataset.rename_column("label", "labels")

train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

# =====================
# 3. Load Transformer Model
# =====================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)
model.to(device)

# =====================
# 4. Metrics
# =====================

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # FIX: eval_pred contains numpy arrays, not tensors — use np.argmax directly
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy":  round(acc, 4),
        "f1":        round(f1, 4),
        "precision": round(precision, 4),
        "recall":    round(recall, 4)
    }

# =====================
# 5. Training Arguments
# =====================

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1",      # FIX: pick best checkpoint by F1, not just loss
    report_to="none"                 # FIX: suppress wandb/tensorboard warnings
)

# =====================
# 6. Trainer
# =====================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# =====================
# 7. Train Model
# =====================

trainer.train()

# =====================
# 8. Evaluate on Test Set
# =====================

# FIX: was passing test_dataset as positional arg — must be keyword arg
results = trainer.evaluate(eval_dataset=test_dataset)
print("\n===== Test Set Results =====")
for k, v in results.items():
    print(f"  {k}: {v}")

# =====================
# 9. Predict & Display Sample Reviews
# =====================

label_map = {0: "Negative ❌", 1: "Positive ✅"}

def predict_review(text: str) -> None:
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()[0]
        pred   = int(np.argmax(probs))

    print(f"\nReview   : {text}")
    print(f"Predicted: {label_map[pred]}")
    print(f"Confidence → Negative: {probs[0]:.3f} | Positive: {probs[1]:.3f}")

# Sample predictions
predict_review("This product is absolutely amazing and works perfectly!")
predict_review("Terrible quality, broke after one day. Total waste of money.")
predict_review("It's okay, nothing special but does the job.")